## DATA 37711 Homework 3
### Glen Morgenstern

## Problem 1
### ai.
Poisson gives us matching support. Y takes in values {0, 1, 2, ...} and Poisson is supported on exactly those values. It's also the canoncial model for count data and has a single parameter so we only need to model the conditional mean.

### aii.
The Poisson rate $\mu$ must be strictly positive, but $f(X) = \sum_j \lambda_j h_j(X)$ is a sum of regression trees and can take any real value. The link function must therefore map $\mathbb{R} \to (0,\infty)$. The exponential does this, and it is the canonical link for the Poisson.

### 1b.
Under the model, $Y \mid X \sim \text{Poi}(\mu)$ with $\mu = \exp(f(X))$, so
$$
P(Y = y \mid X = x) = \frac{\mu^{y} e^{-\mu}}{y!} = \frac{\exp\!\big(y f(x)\big)\exp\!\big(-e^{f(x)}\big)}{y!}.
$$
The negative log likelihood is
$$
-\log P(Y \mid X) = e^{f(X)} - Y f(X) + \log(Y!).
$$
The term $\log(Y!)$ does not depend on $f$, so it can be dropped. The per-example loss is therefore
$$
L\big(f(x), Y\big) = e^{f(x)} - Y f(x),
$$
and we fit $f$ by minimizing the empirical risk
$$
\hat{R}(f) = \frac{1}{n}\sum_{i=1}^{n}\Big( e^{f(x_i)} - y_i f(x_i) \Big).
$$
This loss is convex in $f(x)$.

### 1c.
Differentiating per-example loss $L(f(x), Y) = e^{f(x)} - Y f(x)$ with respect to $f(x)$ gives
$$
\frac{dL}{df(x)} = e^{f(x)} - Y.
$$
This is the predicted mean $\hat{\mu} = e^{f(x)}$ minus the observed count $Y$. The negative gradient, $Y - e^{f(x)}$, is therefore the residual on the mean scale.

### 1d.
First order boosting is gradient descent in function sapce. We build f greedily, one tree at a time.

1. Initialize with the best constant, $f_0(x) = \log \bar{y}$, where $\bar{y} = \frac{1}{n}\sum_i y_i$ (this minimizes $\sum_i e^{c} - y_i c$ over constants $c$)
2. For $j = 1, \dots, J$:
  - Compute negative gradients at current model:
$$
r_i^{(j)} = -\frac{dL}{df(x_i)}\bigg|_{f = f_{j-1}} = y_i - e^{f_{j-1}(x_i)}.
$$
  - Fit a regression tree $h_j$ to $\{(x_i, r_i^{(j)})\}_{i=1}^n$ by ordinary least squares, i.e.
$$
h_j = \arg\min_{h} \sum_{i=1}^n \big( r_i^{(j)} - h(x_i) \big)^2 .
$$
  - Choose the step size $\lambda_j$, either as a fixed small learning rate or by line search
$$
\lambda_j = \arg\min_{\lambda} \sum_{i=1}^n L\big(f_{j-1}(x_i) + \lambda h_j(x_i),\, y_i\big).
$$
  - Update $f_j(x) = f_{j-1}(x) + \lambda_j h_j(x)$.

3. The final model is $f(x) = f_0(x) + \sum_{j=1}^J \lambda_j h_j(x)$, with predicted mean $\hat{\mu}(x) = e^{f(x)}$.

## Problem 2a.

Let $f_1,\dots,f_k$ be the models fit on the training set, one for each hyperparameter choice, and assume the loss satisfies $\ell \in [0,1]$. Let $R(f) = \mathbb{E}[\ell(f(X),Y)]$ be the true risk and $\hat R_{\text{test}}(f) = \frac{1}{l}\sum_{m=1}^{l}\ell(f(X_m),Y_m)$ the risk on the test set, which is drawn i.i.d. and independently of the training set. Let $\hat f = \arg\min_i \hat R_{\text{test}}(f_i)$ be the selected model.

Claim. With probability at least $1-\delta$ over the test set,
$$
\big|\hat R_{\text{test}}(\hat f) - R(\hat f)\big| \le \sqrt{\frac{\log(2k/\delta)}{2l}}.
$$

Proof. Condition on the training set, so that $f_1,\dots,f_k$ are fixed. For each fixed $i$, the test losses $\ell(f_i(X_m),Y_m)$ are i.i.d., lie in $[0,1]$, and have mean $R(f_i)$. Hoeffding's inequality gives, for any $\epsilon>0$,
$$
P\big(|\hat R_{\text{test}}(f_i) - R(f_i)| > \epsilon\big) \le 2e^{-2l\epsilon^2}.
$$
The selected model $\hat f$ depends on the test set, so Hoeffding cannot be applied to it directly. Instead we control all $k$ models at once with a union bound:
$$
P\big(\exists\, i:\ |\hat R_{\text{test}}(f_i) - R(f_i)| > \epsilon\big) \le 2k\,e^{-2l\epsilon^2}.
$$
Setting the right-hand side equal to $\delta$ and solving for $\epsilon$ gives $\epsilon = \sqrt{\log(2k/\delta)/(2l)}$. Hence with probability at least $1-\delta$, \emph{every} $f_i$, and in particular $\hat f$, satisfies the bound. Since this holds conditionally on every training set, it holds unconditionally. $\qquad\blacksquare$

## 2b

The model's test error will be optimistically biased. If we choose our model based on minimizing test risk, then we will see our answer is partly luck. The model whose noise happened to fall favorably on this particular test will get chosen. The bias will grow with k relative to l. Each $\hat R_{test}(f_i)$ fluctuates around $R(f_i)$ at scale about $1/\sqrt(l)$. When k is large relative to l, the expected size of favoarable noise is not small, and the minimum can be driven down by noise alone. The test score will say little about the model's real performance.

## 2c

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

SEED = 0
df = pd.read_csv("winequality-white.csv", sep=";")
X, y = df.drop(columns="quality").values, df["quality"].values